## F3: NLP Annotation

This notebook annotates the F2 VOCAB table with linguistic features from NLTK, producing a dictionary that downstream stages will use for filtering, normalization, and grouping. TOKEN is unchanged from F2 but re-saved to `data/F3/` so each pipeline stage's outputs are self-contained.

### Conceptual goal

F2 produced VOCAB as a frequency dictionary where every distinct word-form has its own corpus count. F3 extends each entry with annotations describing the word's grammatical and morphological identity. These annotations don't alter the token stream; they enrich how it can be sliced. After F3, a single VOCAB row tells you not just that a term appears in Swift, but what kind of word it is, what its root form is, and whether it carries topical signal or is a structural function word.

The assignment doc places stopword flags, POS aggregations, stems, and lemmas explicitly under F3, so the boundary between "constructing the standard tables" (F2) and "annotating with NLP features" (F3) is drawn at this notebook.

### Annotations added to VOCAB

**`stop`** — boolean flag from NLTK's English stopword list. A simple lookup that marks function words ("the", "of", "and") for downstream filtering. Implemented via building a DataFrame of stopwords, mapping onto VOCAB.term_str, and filling missing values with 0.

**`p_stem`** — Porter stem of `term_str`. Applied with `nltk.stem.porter.PorterStemmer`. Porter is rule-based and aggressive, normalizing 64% of terms in the corpus, often producing non-word tokens (`happy > happi`, `little > littl`, `studies > studi`). Useful when the goal is collapsing inflectional variants without caring about linguistic accuracy.

**`pos_max`** — the most common Penn Treebank POS tag for each term across the corpus. Computed by aggregating per-token POS tags (which already exist on TOKEN from F2) up to the term level via `groupby(['term_str','pos']).size().unstack().idxmax(axis=1)`. This makes POS a property of the dictionary entry itself, not just individual tokens. It also enables proper lemmatization in the next step.

**`wn_pos`** — WordNet POS tag derived from `pos_max`. NLTK's lemmatizer requires WordNet's four-tag system (`'n'`, `'v'`, `'a'`, `'r'`), but our POS tags are Penn Treebank (NN, VBD, JJR, etc.). A simple mapping function translates by Penn tag's first letter: N>n, V>v, J>a, R>r, everything else falls back to 'n'.

**`lemma`** — WordNet lemma of `term_str`, computed using `wn_pos` as a hint. Lemma modifies only 32% of terms but handles morphological irregulars Porter cannot: `went > go`, `brought > bring`, `better > good`. Where Porter normalizes broadly but coarsely, WordNet normalizes selectively but linguistically.

### Key observation: Porter vs WordNet

The two normalization strategies represent different philosophical commitments. Porter is mechanical — strip recognizable suffixes by rule, regardless of whether the result is a real word. WordNet is dictionary-based — only modify a term if it's a known inflected form of another dictionary entry, then return the canonical form. The 64% vs 32% modification rate quantifies this difference. Inspecting cases where the two disagree (`went/went/go`, `better/better/good`, `studies/studi/study`) makes the tradeoff concrete and gives F4/F5 analyses a choice point: aggressive collapsing via Porter, or linguistically-precise collapsing via lemma.

### Output

- **VOCAB** (`data/F3/VOCAB.csv`): F2 columns plus `stop`, `p_stem`, `pos_max`, `wn_pos`, `lemma`.
- **TOKEN** (`data/F3/TOKEN.csv`): unchanged from F2, re-saved for stage continuity.

### Notes for downstream stages

The empty-string row at `term_id = 0` carries `pos_max = NaN` (no tokens with that term_str were tagged with a POS by NLTK — they were all punctuation that got stripped). It will continue to be filtered at analysis time. Function words occasionally get spurious lemmas (e.g., `as > a`) because WordNet has poor coverage of non-content words and `wn_pos` defaults them to 'n' — this is harmless because they're filtered as stopwords downstream. F4 will use VOCAB's `pos_max` for term-level analysis, `stop` for filtering, and either `p_stem` or `lemma` (analyst's choice) as the normalization key for TFIDF aggregation.

## Setup

In [57]:
import pandas as pd
import numpy as np
import os
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

F2_path = 'data/F2'
F3_path = 'data/F3'
os.makedirs(F3_path, exist_ok=True)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']


## Load F2 Data

In [58]:
LIB = pd.read_csv(f"{F2_path}/../F1/LIB.csv", index_col=OHCO[0])
TOKEN = pd.read_csv(f"{F2_path}/TOKEN.csv", index_col=OHCO, keep_default_na = False, na_values = [])
VOCAB = pd.read_csv(f"{F2_path}/VOCAB.csv", index_col='term_id', keep_default_na = False, na_values = [])

print(f"LIB: {len(LIB)} works")
print(f"TOKEN: {len(TOKEN):,} tokens")
print(f"VOCAB: {len(VOCAB):,} terms")

LIB: 49 works
TOKEN: 284,077 tokens
VOCAB: 15,394 terms


In [59]:
VOCAB.head()

,term_str,n,num
term_id,,,
0,,117,0
1,0s,1,1
2,1,2,1
3,10,3,1
4,100000,1,1


## Add stopwords

In [60]:
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns = ['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw.dummy = 1

VOCAB['stop'] = VOCAB.term_str.map(sw.dummy)
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')

print(f"Stopwords matched: {VOCAB['stop'].sum()} terms")
VOCAB[VOCAB['stop'] == 1].sample(10)

Stopwords matched: 131 terms


,term_str,n,num,stop
term_id,,,,
15089,who,866,0,1
9224,now,384,0,1
6529,hers,1,0,1
1492,between,141,0,1
3371,d,10,0,1
14327,under,244,0,1
6575,himself,195,0,1
1373,been,590,0,1
6574,him,704,0,1


## Add Porter Stems

In [61]:
stemmer = PorterStemmer()
VOCAB['p_stem'] = VOCAB.term_str.apply(stemmer.stem)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem
term_id,,,,,
8543,meditations,1,0,0,medit
15142,wince,1,0,0,winc
7087,indian,10,0,0,indian
4218,downright,2,0,0,downright
14036,transitory,5,0,0,transitori
760,annum,10,0,0,annum
3411,daring,5,0,0,dare
4915,exchange,14,0,0,exchang
967,artificial,4,0,0,artifici


## Add pos_max (mos common POS per term)

In [62]:
pos_max = (TOKEN.groupby(['term_str','pos']).size()
           .unstack(fill_value=0)
           .idxmax(axis=1)
           .rename('pos_max'))

VOCAB = VOCAB.merge(pos_max, left_on = 'term_str',right_index = True, how = 'left')

VOCAB.pos_max.value_counts().head(15)

pos_max
NN     5823
JJ     2123
NNP    1637
NNS    1461
VB      922
VBG     737
VBN     706
VBD     555
RB      439
VBZ     309
VBP     192
CD      115
JJS     109
IN      106
JJR      47
Name: count, dtype: int64

## Add WordNet lemma

In [63]:
def penn_to_wordnet(tag):
    if not isinstance(tag, str):
        return 'n'
    first = tag[0]
    if first == 'V': return 'v'
    if first == 'J': return 'a'
    if first == 'R': return 'r'
    return 'n'

VOCAB['wn_pos'] = VOCAB.pos_max.apply(penn_to_wordnet)

lemmatizer = WordNetLemmatizer()
VOCAB['lemma'] = VOCAB.apply(lambda r: lemmatizer.lemmatize(r.term_str, r.wn_pos), axis = 1)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem,pos_max,wn_pos,lemma
term_id,,,,,,,,
666,amaranth,1,0,0,amaranth,NN,n,amaranth
12600,snail,2,0,0,snail,NN,n,snail
5570,foresaw,1,0,0,foresaw,VBD,v,foresee
4082,dissent,1,0,0,dissent,NN,n,dissent
9927,perpetual,36,0,0,perpetu,JJ,a,perpetual
815,apollonius,1,0,0,apolloniu,NNP,n,apollonius
522,agesilaus,1,0,0,agesilau,NNP,n,agesilaus
5835,gaping,3,0,0,gape,NN,n,gaping
350,act,43,0,0,act,VB,v,act


In [64]:
diff_lemma = (VOCAB.term_str != VOCAB.lemma).sum()
diff_stem = (VOCAB.term_str != VOCAB.p_stem).sum()
print(f"Terms whose lemma differs from term_str: {diff_lemma:,} ({diff_lemma/len(VOCAB):.1%})")
print(f"Terms whose stem  differs from term_str: {diff_stem:,} ({diff_stem/len(VOCAB):.1%})")


Terms whose lemma differs from term_str: 4,887 (31.7%)
Terms whose stem  differs from term_str: 9,922 (64.5%)


In [65]:
VOCAB[(VOCAB.p_stem != VOCAB.lemma) & (VOCAB.n > 50)].sample(15)[
    ['term_str','n','pos_max','p_stem','lemma']
]

,term_str,n,pos_max,p_stem,lemma
term_id,,,,,
9042,necessary,78,JJ,necessari,necessary
14226,twenty,71,JJ,twenti,twenty
15106,why,213,NNP,whi,why
4167,does,79,VBZ,doe,do
7638,kept,70,VBD,kept,keep
5497,following,54,JJ,follow,following
12145,sent,92,VBD,sent,send
7714,knowledge,63,NN,knowledg,knowledge
663,always,155,RB,alway,always


In [66]:
TOKEN.to_csv(f"{F3_path}/TOKEN.csv")
VOCAB.to_csv(f"{F3_path}/VOCAB.csv")